# 1.0 — Setup & Imports

Mount Google Drive (when running in Colab), install required packages, and import all libraries
used throughout the notebook. Identical to the baseline uncertainty-comparison notebook.

In [1]:
import os
import sys
import subprocess

try:
    import xlsxwriter
except ImportError:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlsxwriter", "openpyxl"], check=True)
    except Exception as _pip_ex:
        print(f'Notice: optional pip install skipped ({_pip_ex})')

In [2]:
from scipy.io import loadmat
import io
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from openpyxl import load_workbook

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)


Python: 3.13.16
TensorFlow: 2.21.0
NumPy: 2.1.3
Pandas: 2.3.3


# 2.0 — Configuration

All project-wide constants, file paths, and hyperparameters. `DAPM_DIR` points to the saved
DAPM artifacts (encoder / classifier / diffusion weights + JSON configs) produced by the DAPM
training notebook; `MODEL_DIR` points to the underlying baseline `.keras` backbones that the
DAPM feature extractors are built from. All output filenames are prefixed `dapm_conformal_`
so they never collide with the plain-model conformal run or the DAPM t-test run.

In [3]:
# ── Reproducibility ──────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ── Environment & Paths ─────────────────────────────────────────────────────────
import glob

IN_KAGGLE = Path('/kaggle').exists() or Path('/kaggle/working').exists()
IN_COLAB  = False

if IN_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    OUTPUT_DIR   = PROJECT_ROOT / 'results/conformal_uncertainty_results'
    RESULTS_DIR  = PROJECT_ROOT / 'results'
    MODEL_DIR    = Path('/kaggle/working/models')
    dapm_cands = list(Path('/kaggle/kin').glob('**/dapm_full_artifacts')) or list(Path('/kaggle/kin').glob('**/*_dapm_full_encoder.weights.h5')) or list(Path('/kaggle/working/models').glob('**/*_dapm_full_encoder.weights.h5'))
    if dapm_cands:
        DAPM_DIR = dapm_cands[0] if dapm_cands[0].is_dir() else dapm_cands[0].parent
    else:
        DAPM_DIR = Path('/kaggle/working/models')
else:
    PROJECT_ROOT = Path('.')
    OUTPUT_DIR   = Path('./results/conformal_uncertainty_results')
    RESULTS_DIR  = Path('./results')
    MODEL_DIR    = Path('./models')
    DAPM_DIR     = Path('./models')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset path resolution ──────────────────────────────────────────────────
DATA_KEY  = "indian_pines"
LABEL_KEY = "indian_pines_gt"
DATASET_NAME = "Indian Pines"

mat_candidates = (
    list(Path("/kaggle/kin").glob("**/Indian_pines.mat"))
    or list(Path("/kaggle/kin").glob("**/k001.mat"))
    or list(Path("data").glob("**/Indian_pines.mat"))
    or list(Path("examples/indian_pines_uncertainty_quantification/data").glob("**/Indian_pines.mat"))
    or list(Path("../data").glob("**/Indian_pines.mat"))
)
if mat_candidates:
    DATA_FILE  = mat_candidates[0]
    LABEL_FILE = DATA_FILE.parent / "Indian_pines_gt.mat"
    if not LABEL_FILE.exists():
        gt_cands = (
            list(Path("/kaggle/kin").glob("**/Indian_pines_gt.mat"))
            or list(Path("/kaggle/kin").glob("**/k002.mat"))
            or [p for p in [Path("data/Indian_pines_gt.mat"), Path("../data/Indian_pines_gt.mat")] if p.exists()]
        )
        LABEL_FILE = gt_cands[0] if gt_cands else DATA_FILE.parent / "Indian_pines_gt.mat"
    DATA_DIR = DATA_FILE.parent
else:
    local_candidates = [
        Path("data"),
        Path("examples/indian_pines_uncertainty_quantification/data"),
        Path("../data"),
    ]
    DATA_DIR = next((p for p in local_candidates if (p / "Indian_pines.mat").exists()), Path("data"))
    DATA_FILE  = DATA_DIR / "Indian_pines.mat"
    LABEL_FILE = DATA_DIR / "Indian_pines_gt.mat"


def resolve_model_file(model_dir: Path, obfuscated_name: str, original_name: str) -> Path:
    candidates = [
        model_dir / original_name,
        model_dir / obfuscated_name,
        Path(f'/kaggle/working/models/{original_name}'),
        Path(f'/kaggle/kin/{original_name}'),
        Path(f'/kaggle/kin/{obfuscated_name}'),
        Path(f'models/{original_name}'),
    ]
    for c in candidates:
        if c.exists():
            return c
    found = glob.glob(f'/kaggle/kin/**/{original_name}', recursive=True)
    if found:
        return Path(found[0])
    found_obf = glob.glob(f'/kaggle/kin/**/{obfuscated_name}', recursive=True)
    if found_obf:
        return Path(found_obf[0])
    local_found = glob.glob(f'**/{original_name}', recursive=True)
    if local_found:
        return Path(local_found[0])
    return candidates[0]

MODEL_KEYS = ['AlexNet_CNN', 'GFNet', 'ViT_UNet']
MODEL_FILES = {
    'AlexNet_CNN': resolve_model_file(MODEL_DIR, 'k003.keras', 'AlexNet_CNN_best.keras'),
    'GFNet':       resolve_model_file(MODEL_DIR, 'k004.keras', 'GFNet_best.keras'),
    'ViT_UNet':    resolve_model_file(MODEL_DIR, 'k005.keras', 'ViT_UNet_best.keras'),
}
MODEL_NAME_MAP = {
    'AlexNet_CNN': 'AlexNet',
    'GFNet':       'GFNet',
    'ViT_UNet':    'ViT',
}

# ── Image / patch geometry ───────────────────────────────────
H, W, B = 145, 145, 220
PERCENTILE_CLIP = (2.0, 98.0)
PATCH_SIZE = 9

# ── Train / calibration split ────────────────────────────────
TRAIN_PERCENT          = 0.75
CALIB_FRACTION_OF_TEST = 0.50

# ── Method hyperparameters ───────────────────────────────────
ALPHA      = 0.05
RAPS_LAM   = 0.01
RAPS_K_REG = 1
N_CLUSTERS = 4
BATCH_SIZE = 128
EPS        = 1e-12

EXCEL_PATH           = OUTPUT_DIR / 'dapm_conformal_reports_all_models.xlsx'
SUMMARY_CSV_PATH     = OUTPUT_DIR / 'dapm_conformal_summary_metrics.csv'
PER_CLASS_CSV_PATH   = OUTPUT_DIR / 'dapm_conformal_per_class_coverage.csv'
RUN_CONFIG_JSON_PATH = OUTPUT_DIR / 'dapm_conformal_run_config.json'

for k, v in MODEL_FILES.items():
    print(f'{k}: {v}')
print('DAPM artifacts dir:', DAPM_DIR)
print('Workbook:', EXCEL_PATH)


AlexNet_CNN: /kaggle/working/models/AlexNet_CNN_best.keras
GFNet: /kaggle/working/models/GFNet_best.keras
ViT_UNet: /kaggle/working/models/ViT_UNet_best.keras
DAPM artifacts dir: /kaggle/working/models
Workbook: /kaggle/working/results/conformal_uncertainty_results/dapm_conformal_reports_all_models.xlsx


# 3.0 — Environment Guards

Validates that all required data files, baseline backbone files, and DAPM artifact configs
exist before the pipeline runs. Raises clear errors if anything is missing.

In [4]:
def assert_environment_and_files():
    """Raise informative errors if required data, backbone, or DAPM artifact files are absent."""
    if not DATA_FILE.exists() or not LABEL_FILE.exists():
        raise FileNotFoundError(
            f'Data files missing:\n- {DATA_FILE}\n- {LABEL_FILE}'
        )

    missing_models = [str(p) for p in MODEL_FILES.values() if not p.exists()]
    if missing_models:
        msg = 'Missing baseline model files:\n' + '\n'.join(missing_models)
        raise FileNotFoundError(msg)

    missing_dapm = [
        str(DAPM_DIR / f'{mk}_dapm_full_config.json')
        for mk in MODEL_KEYS
        if not (DAPM_DIR / f'{mk}_dapm_full_config.json').exists()
    ]
    if missing_dapm:
        msg = 'Missing DAPM artifact configs:\n' + '\n'.join(missing_dapm)
        msg += '\n\nRun the DAPM training notebook first to produce these artifacts.'
        raise FileNotFoundError(msg)

assert_environment_and_files()
print('All required data / backbone / DAPM artifact files are present.')


All required data / backbone / DAPM artifact files are present.


# 4.0 — Data Pipeline

Loads the multispectral image and reference labels from CSV, extracts labeled patches, and
performs train / calibration / evaluation splits. Identical to the baseline uncertainty-comparison
notebook — the same patches feed both the baseline models and the DAPM bundles.

## 4.1 — Data Loading & Patch Extraction

In [5]:
def load_scene_from_mat(data_path, label_path, data_key=None, label_key=None):
    """Load a hyperspectral scene + reference labels from .mat files.
    Target: Indian Pines (145 x 145 x 220, 16 active classes).
    Loads 'Indian_pines.mat' (key: 'indian_pines') and 'Indian_pines_gt.mat' (key: 'indian_pines_gt').
    Normalizes each spectral band to [0, 1] using percentile clipping [2.0, 98.0].
    """
    data_key  = data_key or "indian_pines"
    label_key = label_key or "indian_pines_gt"
    data_mat  = loadmat(data_path)
    label_mat = loadmat(label_path)

    def _first_array(mat_dict, key):
        if key is not None and key in mat_dict:
            return mat_dict[key]
        candidates = [v for k, v in mat_dict.items() if not k.startswith("__")]
        if not candidates:
            raise ValueError(f"No data arrays found in {mat_dict.keys()}")
        return candidates[0]

    x = _first_array(data_mat, data_key).astype(np.float32)
    y = _first_array(label_mat, label_key).astype(np.int32)

    if x.ndim != 3:
        raise ValueError(f"Expected a (H, W, Bands) cube, got shape {x.shape}")
    if y.shape != x.shape[:2]:
        raise ValueError(f"Label shape {y.shape} does not match image shape {x.shape[:2]}")

    height, width, bands = x.shape

    x_norm = np.empty_like(x, dtype=np.float32)
    for b in range(bands):
        band = x[:, :, b]
        lo   = np.percentile(band, PERCENTILE_CLIP[0])
        hi   = np.percentile(band, PERCENTILE_CLIP[1])
        denom = max(hi - lo, 1e-8)
        band_clipped = np.clip(band, lo, hi)
        x_norm[:, :, b] = (band_clipped - lo) / denom

    return x_norm, y, height, width, bands

def extract_labeled_patches(x, y, patch_size=9):
    """Extract (patch_size x patch_size) spatial patches around every labelled pixel (y > 0)."""
    pad   = patch_size // 2
    x_pad = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode="edge")

    coords  = np.argwhere(y > 0)
    patches = np.empty((coords.shape[0], patch_size, patch_size, x.shape[-1]), dtype=np.float32)
    labels  = np.empty((coords.shape[0],), dtype=np.int32)

    for i, (r, c) in enumerate(coords):
        patches[i] = x_pad[r:r + patch_size, c:c + patch_size, :]
        labels[i]  = int(y[r, c]) - 1  # 1-indexed to 0..K-1

    return patches, labels, coords

def extract_labeled_patches_with_coords(x_img, y_img, patch_size=9):
    return extract_labeled_patches(x_img, y_img, patch_size)

def split_calib_eval(x_test, y_test, seed=42, calib_fraction=0.5):
    """Split a test pool into calibration and evaluation sets (stratified if possible)."""
    test_size = 1.0 - calib_fraction
    try:
        x_cal, x_eval, y_cal, y_eval = train_test_split(
            x_test, y_test, test_size=test_size, random_state=seed, stratify=y_test,
        )
    except ValueError:
        x_cal, x_eval, y_cal, y_eval = train_test_split(
            x_test, y_test, test_size=test_size, random_state=seed, stratify=None,
        )
    return x_cal, x_eval, y_cal, y_eval


## 4.2 — Build Splits

In [6]:
x_img, y_img, H, W, B = load_scene_from_mat(DATA_FILE, LABEL_FILE)
X_all, y_all, coords_all = extract_labeled_patches(x_img, y_img, PATCH_SIZE)
num_classes  = int(np.unique(y_all).size)

# FIX C1/C3/C4 — shared train / val / cal / eval split (identical code in every notebook) ──
# SPLIT_MODE = "spatial_block": whole spatial blocks go to one role and training
#   pixels whose 9x9 patch would contain a cal/eval pixel are dropped, so no
#   labelled test pixel is ever seen inside a training patch.
# SPLIT_MODE = "random": legacy stratified pixel-wise split (spatially leaky).
import numpy as _np_split
SPLIT_MODE        = globals().get("SPLIT_MODE", "spatial_block")
SPLIT_SEED        = 42
SPLIT_BLOCK       = 10
SPLIT_TRAIN_FRAC  = 0.75   # fraction of labelled pixels assigned to train+val
SPLIT_VAL_FRAC    = 0.20   # fraction of train+val used for validation / checkpointing
SPLIT_CAL_FRAC    = 0.50   # fraction of the held-out pool used for conformal calibration


def make_shared_split(y_img, patch_size=9, mode=None, seed=SPLIT_SEED,
                      train_frac=SPLIT_TRAIN_FRAC, val_frac=SPLIT_VAL_FRAC,
                      cal_frac=SPLIT_CAL_FRAC, block=SPLIT_BLOCK):
    """Return dict of index arrays (into np.argwhere(y_img > 0) order):
    train, val, cal, eval, test (= cal U eval)."""
    mode   = mode or SPLIT_MODE
    coords = _np_split.argwhere(y_img > 0)
    labels = y_img[y_img > 0].astype(int) - 1
    n, K   = len(labels), int(labels.max()) + 1
    rng    = _np_split.random.default_rng(seed)
    roles  = _np_split.full(n, -1)                       # 0 train, 1 val, 2 cal, 3 eval
    held   = 1.0 - train_frac
    frac   = _np_split.array([train_frac * (1 - val_frac), train_frac * val_frac,
                              held * cal_frac, held * (1 - cal_frac)])
    if mode == "random":
        from sklearn.model_selection import train_test_split as _tts
        idx = _np_split.arange(n)
        def _strat(a):
            cnt = _np_split.bincount(labels[a], minlength=K)
            return labels[a] if cnt[cnt > 0].min() >= 2 else None
        tv, te = _tts(idx, train_size=train_frac, random_state=seed, stratify=_strat(idx))
        tr, va = _tts(tv, test_size=val_frac, random_state=seed, stratify=_strat(tv))
        ca, ev = _tts(te, test_size=1 - cal_frac, random_state=seed, stratify=_strat(te))
        for r, a in enumerate([tr, va, ca, ev]):
            roles[a] = r
    else:
        bid = (coords[:, 0] // block) * 100000 + (coords[:, 1] // block)
        ub, inv = _np_split.unique(bid, return_inverse=True)
        cnt = _np_split.zeros((len(ub), K))
        _np_split.add.at(cnt, (inv, labels), 1)
        target = frac[None, :] * cnt.sum(0)[:, None]          # (K, 4)
        have   = _np_split.zeros((K, 4))
        order  = rng.permutation(len(ub))
        # rare classes first so each role receives some of them
        rarity = (cnt / _np_split.maximum(cnt.sum(0, keepdims=True), 1)).max(1)
        order  = order[_np_split.argsort(-rarity[order], kind="stable")]
        for b in order:
            w    = cnt[b]
            need = ((target - have) / _np_split.maximum(target, 1)) * w[:, None]
            role = int(_np_split.argmax(need.sum(0)))
            have[:, role] += w
            roles[inv == b] = role
        # drop training/validation pixels whose patch would contain a held-out pixel
        pad  = patch_size // 2
        grid = _np_split.zeros(y_img.shape, dtype=bool)
        hc   = coords[roles >= 2]
        grid[hc[:, 0], hc[:, 1]] = True
        from scipy.ndimage import maximum_filter as _mf
        near = _mf(grid.astype(_np_split.uint8), size=2 * pad + 1) > 0
        drop = (roles <= 1) & near[coords[:, 0], coords[:, 1]]
        roles[drop] = -1
    out = {k: _np_split.where(roles == r)[0] for r, k in enumerate(["train", "val", "cal", "eval"])}
    out["test"] = _np_split.sort(_np_split.concatenate([out["cal"], out["eval"]]))
    out["coords"], out["labels"], out["mode"] = coords, labels, mode
    return out


def check_split_alignment(split, ns):
    """Fail loudly if the notebook's labelled-patch order differs from the shared split order
    (row-major over y > 0). Any 1-D label vector of the same length is compared up to relabelling."""
    lab = split["labels"]; n = len(lab); checked = []
    for name in ("y", "Y", "y_all", "y_labels", "y_labeled"):
        v = ns.get(name)
        if isinstance(v, _np_split.ndarray) and v.ndim == 1 and len(v) == n:
            pairs = set(zip(lab.tolist(), _np_split.asarray(v).astype(int).tolist()))
            if len(pairs) != len(set(lab.tolist())) or len({b for _, b in pairs}) != len(pairs):
                raise RuntimeError(f"shared split is not aligned with label vector '{name}'")
            checked.append(name)
    print("split alignment verified against:", checked or "nothing (no 1-D label vector in scope)")


def describe_split(split, num_classes):
    import pandas as _pd
    rows = {k: _np_split.bincount(split["labels"][split[k]], minlength=num_classes)
            for k in ["train", "val", "cal", "eval"]}
    df = _pd.DataFrame(rows); df.index.name = "class"
    print(f"Split mode: {split['mode']}  |  sizes:", {k: int(v.sum()) for k, v in rows.items()})
    return df


# FIX C1: shared split -> identical cal / eval pixels for every model and notebook
SPLIT = make_shared_split(y_img, patch_size=PATCH_SIZE)
check_split_alignment(SPLIT, globals())
display(describe_split(SPLIT, num_classes))
x_cal,  y_cal  = X_all[SPLIT["cal"]],  y_all[SPLIT["cal"]]
x_eval, y_eval = X_all[SPLIT["eval"]], y_all[SPLIT["eval"]]

print('x_img:', x_img.shape, ' y_img:', y_img.shape)
print('X_all:', X_all.shape, ' y_all:', y_all.shape, ' num_classes:', num_classes)
print('x_cal:', x_cal.shape, ' x_eval:', x_eval.shape)


split alignment verified against: ['y_all']
Split mode: spatial_block  |  sizes: {'train': 3042, 'val': 985, 'cal': 1426, 'eval': 1385}


,train,val,cal,eval
class,,,,
0,0,3,15,0
1,403,152,197,188
2,208,40,110,111
3,79,11,31,54
4,168,54,72,77
5,132,90,106,91
6,14,14,0,0
7,109,60,90,90
8,20,0,0,0


x_img: (145, 145, 220)  y_img: (145, 145)
X_all: (10249, 9, 9, 220)  y_all: (10249,)  num_classes: 16
x_cal: (1426, 9, 9, 220)  x_eval: (1385, 9, 9, 220)


# 5.0 — Custom Keras Layers

Registers all project-specific Keras layers required to deserialise the saved baseline models
(`PatchExtractor`, `PatchPositionEncoder`, `GlobalFilterLayer`, `PatchEncoderWithCLS`) plus the
`Sampling` layer used inside the DAPM VAE encoder. These must be defined before any
`keras.models.load_model` call or DAPM sub-network construction.

## 5.1 — Backbone Deserialisation Layers

In [7]:
try:
    keras.config.enable_unsafe_deserialization()
except Exception:
    pass

@tf.keras.utils.register_keras_serializable()
class PatchExtractor(layers.Layer):
    """Extract non-overlapping image patches using tf.image.extract_patches."""

    def __init__(self, patch_size=3, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patches  = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        batch    = tf.shape(images)[0]
        n_patches = tf.shape(patches)[1] * tf.shape(patches)[2]
        patch_dim = tf.shape(patches)[-1]
        return tf.reshape(patches, [batch, n_patches, patch_dim])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'patch_size': self.patch_size})
        return cfg


@tf.keras.utils.register_keras_serializable()
class PatchPositionEncoder(layers.Layer):
    """Project patches to an embedding dimension and add positional encodings."""

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection         = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patches):
        pos = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(pos)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return cfg


@tf.keras.utils.register_keras_serializable()
class GlobalFilterLayer(layers.Layer):
    """GFNet global filter layer: learnable frequency-domain filtering via FFT."""

    def __init__(self, token_side, **kwargs):
        super().__init__(**kwargs)
        self.token_side = token_side

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.w_real = self.add_weight(
            name='w_real', shape=(self.token_side, self.token_side, channels),
            initializer='glorot_uniform', trainable=True,
        )
        self.w_imag = self.add_weight(
            name='w_imag', shape=(self.token_side, self.token_side, channels),
            initializer='zeros', trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        b  = tf.shape(x)[0]
        c  = tf.shape(x)[-1]
        x2 = tf.reshape(x, [b, self.token_side, self.token_side, c])
        # FIX C5: fft2d acts on the two innermost axes -> put channels first so the
        # filter runs over the (token_row, token_col) grid.
        x2  = tf.transpose(x2, [0, 3, 1, 2])
        x_fft = tf.signal.fft2d(tf.cast(x2, tf.complex64))
        w  = tf.transpose(tf.complex(self.w_real, self.w_imag), [2, 0, 1])
        x_f = x_fft * w
        x_i = tf.transpose(tf.math.real(tf.signal.ifft2d(x_f)), [0, 2, 3, 1])
        return tf.reshape(x_i, [b, self.token_side * self.token_side, c])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'token_side': self.token_side})
        return cfg


@tf.keras.utils.register_keras_serializable()
class PatchEncoderWithCLS(layers.Layer):
    """Patch encoder that prepends a learnable [CLS] token and adds positional embeddings."""

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection         = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches + 1, output_dim=projection_dim)

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            name='cls_token', shape=(1, 1, self.projection_dim),
            initializer='zeros', trainable=True,
        )
        super().build(input_shape)

    def call(self, patches):
        batch      = tf.shape(patches)[0]
        patch_proj = self.projection(patches)
        cls_tokens = tf.repeat(self.cls_token, repeats=batch, axis=0)
        x   = tf.concat([cls_tokens, patch_proj], axis=1)
        pos = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        return x + self.position_embedding(pos)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return cfg


CUSTOM_OBJECTS = {
    'PatchExtractor':       PatchExtractor,
    'PatchPositionEncoder': PatchPositionEncoder,
    'GlobalFilterLayer':    GlobalFilterLayer,
    'PatchEncoderWithCLS':  PatchEncoderWithCLS,
}

# ── Robust Backbone Loader across Python versions (bypasses Lambda bytecode deserialization) ──
import zipfile

def transformer_block(x, num_heads, projection_dim, mlp_dim, dropout_rate, name_prefix):
    """Standard pre-LN transformer block with multi-head attention and a GELU MLP."""
    y = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_ln1")(x)
    y = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim,
        dropout=dropout_rate, name=f"{name_prefix}_mha",
    )(y, y)
    x = layers.Add(name=f"{name_prefix}_add1")([y, x])

    y = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_ln2")(x)
    y = layers.Dense(mlp_dim, activation=tf.keras.activations.gelu, name=f"{name_prefix}_mlp1")(y)
    y = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop1")(y)
    y = layers.Dense(projection_dim, activation=tf.keras.activations.gelu, name=f"{name_prefix}_mlp2")(y)
    y = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop2")(y)
    return layers.Add(name=f"{name_prefix}_add2")([y, x])


def build_vit_unet_singlehead(
    input_shape=(9, 9, 220), num_classes=16, inner_patch=3,
    projection_dim=256, num_heads=4, transformer_layers=12,
    mlp_multiplier=2, dropout_rate=0.25, head_units=(512, 256, 128, 64),
):
    """Build a ViT single-head classifier matching dias-baseline-k03."""
    num_patches = (input_shape[0] // inner_patch) * (input_shape[1] // inner_patch)

    inputs = keras.Input(shape=input_shape)
    x      = PatchExtractor(inner_patch, name="vit_patch_extractor")(inputs)
    x      = PatchEncoderWithCLS(num_patches, projection_dim, name="vit_patch_encoder")(x)

    block_list = []
    for i in range(transformer_layers):
        x = transformer_block(
            x, num_heads=num_heads, projection_dim=projection_dim,
            mlp_dim=projection_dim * mlp_multiplier,
            dropout_rate=0.1, name_prefix=f"vit_block_{i+1}",
        )
        if i <= transformer_layers // 2:
            block_list.append(x)
        else:
            x = layers.Add(name=f"vit_skip_add_{i+1}")([x, block_list[transformer_layers - i - 1]])

    x         = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_1")(x)
    x         = layers.LayerNormalization(epsilon=1e-6, name="vit_cls_norm")(x)
    cls_token = layers.Lambda(lambda t: t[:, 0, :], name="vit_cls_token")(x)

    y = layers.Dense(head_units[0], activation=tf.keras.activations.gelu, name="vit_head_1")(cls_token)
    y = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_3")(y)
    y = layers.Dense(head_units[1], activation=tf.keras.activations.gelu, name="vit_head_2")(y)
    y = layers.Dense(head_units[2], activation=tf.keras.activations.gelu, name="vit_head_3")(y)
    y = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_5")(y)
    y = layers.Dense(head_units[3], activation=tf.keras.activations.gelu, name="vit_head_4")(y)
    y = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_6")(y)

    outputs = layers.Dense(num_classes, activation="softmax", name="vit_logits")(y)
    return keras.Model(inputs, outputs, name="ViT_UNet_SingleHead")


def load_backbone_model(path, model_key, custom_objects, input_shape=(9, 9, 220), num_classes=16):
    """Load model safely without Python bytecode deserialization crash."""
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Missing base model: {path}")

    if model_key == "ViT_UNet":
        model = build_vit_unet_singlehead(input_shape=input_shape, num_classes=num_classes)
        try:
            model.load_weights(str(path_obj))
        except Exception:
            if zipfile.is_zipfile(path_obj):
                with zipfile.ZipFile(path_obj, "r") as z:
                    if "model.weights.h5" in z.namelist():
                        tmp_w = f"/tmp/{model_key}_weights.h5"
                        with open(tmp_w, "wb") as f_out:
                            f_out.write(z.read("model.weights.h5"))
                        model.load_weights(tmp_w)
                    else:
                        raise
            else:
                raise
        return model
    else:
        return keras.models.load_model(str(path_obj), custom_objects=custom_objects, compile=False, safe_mode=False)


## 5.2 — DAPM VAE Sampling Layer

In [8]:
@tf.keras.utils.register_keras_serializable()
class Sampling(layers.Layer):
    """VAE reparameterisation trick: sample z from N(z_mu, exp(0.5 * z_logvar))."""

    def call(self, inputs):
        z_mu, z_logvar = inputs
        eps = tf.random.normal(shape=tf.shape(z_mu))
        return z_mu + tf.exp(0.5 * z_logvar) * eps


# Extend the registry so it also covers the DAPM sampling layer
CUSTOM_OBJECTS['Sampling'] = Sampling


# 6.0 — DAPM Model Builders & Bundle Loader

Constructs the DAPM sub-networks (feature extractor, VAE encoder, latent classifier, and
conditional diffusion denoiser) and defines `load_dapm_bundle`, which reads a saved model's
JSON config, rebuilds every sub-network, and restores its trained weights. Reused unchanged
from the DAPM full-scene uncertainty notebook.

## 6.1 — Sub-network Builders

In [9]:
def get_feature_extractor(base_model):
    """Wrap the penultimate layer of a Keras model as a frozen feature extractor."""
    penultimate = base_model.layers[-2].output
    feat_model  = keras.Model(
        base_model.input, penultimate,
        name=f'{base_model.name}_feature_extractor'
    )
    feat_model.trainable = False
    return feat_model


def build_dapm_encoder(feature_dim, latent_dim=64, hidden_dim=256):
    """Build the VAE encoder: features → (z_mu, z_logvar, z_sample)."""
    inp     = keras.Input(shape=(feature_dim,), name='enc_feature_in')
    h       = layers.Dense(hidden_dim, activation='relu', name='enc_h1')(inp)
    h       = layers.Dense(hidden_dim, activation='relu', name='enc_h2')(h)
    z_mu    = layers.Dense(latent_dim, name='z_mu')(h)
    z_logvar = layers.Dense(latent_dim, name='z_logvar')(h)
    z       = Sampling(name='z_sample')([z_mu, z_logvar])
    return keras.Model(inp, [z_mu, z_logvar, z], name='dapm_full_encoder')


def build_dapm_classifier(latent_dim, num_classes, hidden_dim=128):
    """Build the latent-space softmax classifier: z → class probabilities."""
    inp = keras.Input(shape=(latent_dim,), name='clf_z_in')
    h   = layers.Dense(hidden_dim, activation='relu', name='clf_h1')(inp)
    out = layers.Dense(num_classes, activation='softmax', name='clf_out')(h)
    return keras.Model(inp, out, name='dapm_full_classifier')


def build_dapm_diffusion(latent_dim, num_classes, T=100, t_embed_dim=32, hidden_dim=256):
    """Build the conditional diffusion denoiser: (z, y_t, guidance, t) → eps_pred."""
    z_in   = keras.Input(shape=(latent_dim,),   name='diff_z_in')
    y_t_in = keras.Input(shape=(num_classes,),  name='diff_y_t')
    f_in   = keras.Input(shape=(num_classes,),  name='diff_guidance')
    t_in   = keras.Input(shape=(1,), dtype='int32', name='diff_t')
    t_emb  = layers.Embedding(input_dim=T + 1, output_dim=t_embed_dim, name='diff_t_embed')(t_in)
    t_emb  = layers.Flatten(name='diff_t_flat')(t_emb)
    x      = layers.Concatenate(name='diff_concat')([z_in, y_t_in, f_in, t_emb])
    x      = layers.Dense(hidden_dim, activation='relu', name='diff_h1')(x)
    x      = layers.Dense(hidden_dim, activation='relu', name='diff_h2')(x)
    eps_pred = layers.Dense(num_classes, activation='linear', name='diff_eps_pred')(x)
    return keras.Model([z_in, y_t_in, f_in, t_in], eps_pred, name='dapm_full_diffusion')


def make_beta_schedule(T, beta_start=1e-4, beta_end=2e-2):
    """Linear beta schedule → (betas, alphas, cumulative alpha_bars)."""
    betas      = np.linspace(beta_start, beta_end, T, dtype=np.float32)
    alphas     = 1.0 - betas
    alpha_bars = np.cumprod(alphas)
    return betas, alphas, alpha_bars


## 6.2 — Bundle Loader

In [10]:
def load_dapm_bundle(model_key):
    """
    Load a complete DAPM bundle for `model_key`.

    Reads the JSON config, loads the base feature extractor, builds all sub-networks,
    performs a warm-up forward pass to materialise shapes, loads saved weights,
    and returns a dict with all components plus the diffusion schedule.
    """
    cfg_path = DAPM_DIR / f'{model_key}_dapm_full_config.json'
    if not cfg_path.exists():
        raise FileNotFoundError(f'Missing config: {cfg_path}')
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))

    feature_dim     = int(cfg['feature_dim'])
    latent_dim      = int(cfg['latent_dim'])
    num_classes_cfg = int(cfg['num_classes'])

    base_model_path = MODEL_FILES[model_key]
    if not base_model_path.exists():
        raise FileNotFoundError(f'Missing local base model: {base_model_path}')

    print(f'  Loading base model from: {base_model_path}')
    base_model = load_backbone_model(
        base_model_path, model_key, CUSTOM_OBJECTS,
        input_shape=(PATCH_SIZE, PATCH_SIZE, B),
        num_classes=num_classes_cfg,
    )
    feature_extractor = get_feature_extractor(base_model)

    encoder    = build_dapm_encoder(
        feature_dim, latent_dim=latent_dim,
        hidden_dim=int(cfg['decoder_hidden_dim'])
    )
    classifier = build_dapm_classifier(
        latent_dim, num_classes_cfg,
        hidden_dim=int(cfg['classifier_hidden_dim'])
    )
    diffusion  = build_dapm_diffusion(
        latent_dim, num_classes_cfg,
        T=int(cfg['diffusion_T']),
        t_embed_dim=int(cfg.get('t_embed_dim', 32)),
        hidden_dim=int(cfg['diffusion_hidden_dim'])
    )

    # Warm-up forward pass to build all layer shapes before loading weights
    feat_dummy            = tf.zeros((1, feature_dim), dtype=tf.float32)
    z_mu_d, z_logvar_d, z_d = encoder(feat_dummy, training=False)
    y_guidance_d          = classifier(z_d, training=False)
    _                     = diffusion(
        [z_d, y_guidance_d, y_guidance_d, tf.ones((1, 1), dtype=tf.int32)],
        training=False
    )

    # Load saved weights
    encoder.load_weights(str(DAPM_DIR / f'{model_key}_dapm_full_encoder.weights.h5'))
    classifier.load_weights(str(DAPM_DIR / f'{model_key}_dapm_full_classifier.weights.h5'))
    diffusion.load_weights(str(DAPM_DIR / f'{model_key}_dapm_full_diffusion.weights.h5'))
    print(f'  Weights loaded for {model_key}')

    betas, alphas, alpha_bars = make_beta_schedule(
        int(cfg['diffusion_T']),
        beta_start=float(cfg['beta_start']),
        beta_end=float(cfg['beta_end'])
    )
    return {
        'cfg':               cfg,
        'feature_extractor': feature_extractor,
        'encoder':           encoder,
        'classifier':        classifier,
        'diffusion':         diffusion,
        'betas':             betas,
        'alphas':            alphas,
        'alpha_bars':        alpha_bars,
        'T':                 int(cfg['diffusion_T']),
        'num_classes':       num_classes_cfg,
    }


# 7.0 — Shared Utilities

Reusable helpers for probability normalisation, conformal metrics, plotting, and inference.
All conformal method builders in Section 8 depend on these.

## 7.1 — Probability & Conformal Metrics Helpers

In [11]:
# Colour palette for class maps
import colorsys

CLASS_COLOR_BASE = [
    '#0000FF', '#00FF00', '#FF0000', '#00FFFF', '#FF00FF',
    '#FFFF00', '#A52A2A', '#FFA500', '#7FFF00', '#8A2BE2',
]
BACKGROUND_COLOR = '#000000'
UNCERTAIN_COLOR  = '#808080'


def _generate_extra_shades(n_extra, hue_offset=0.05, sat=0.85, val=0.95):
    """Bold, highly-saturated extra hues in the same style as CLASS_COLOR_BASE."""
    shades = []
    for i in range(n_extra):
        hue = (hue_offset + i / max(n_extra, 1)) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
        shades.append('#%02x%02x%02x' % (int(r * 255), int(g * 255), int(b * 255)))
    return shades


def get_class_colors(n):
    """Return n visually distinct hex colours; extends past the base 10 with
    additional bold shades generated in the same saturated style."""
    if n <= len(CLASS_COLOR_BASE):
        return CLASS_COLOR_BASE[:n]
    return CLASS_COLOR_BASE + _generate_extra_shades(n - len(CLASS_COLOR_BASE))


def resolve_uncertain_color(class_colors, preferred=UNCERTAIN_COLOR, min_dist=60):
    """Return `preferred` unless it visually clashes with any class colour, in
    which case fall back to a distinct dark slate so 'Uncertain' never collides."""
    def _hex_to_rgb(h):
        h = h.lstrip('#')
        return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    pr, pg, pb = _hex_to_rgb(preferred)
    for c in class_colors:
        cr, cg, cb = _hex_to_rgb(c)
        dist = ((pr - cr) ** 2 + (pg - cg) ** 2 + (pb - cb) ** 2) ** 0.5
        if dist < min_dist:
            return '#2F2F2F'
    return preferred


def normalize_probs(prob, eps=1e-12):
    """Clip, sanitise, and row-normalise a probability matrix."""
    prob = np.asarray(prob, dtype=np.float64)
    prob = np.nan_to_num(prob, nan=0.0, posinf=0.0, neginf=0.0)
    prob = np.clip(prob, 0.0, 1.0)
    rs   = prob.sum(axis=-1, keepdims=True)
    rs   = np.where(rs <= eps, 1.0, rs)
    return prob / rs


def predict_probs(model, x, batch_size=128):
    """Run model inference and return sanitised probability array."""
    prob = model.predict(x, batch_size=batch_size, verbose=0)
    return normalize_probs(prob, eps=EPS)


def safe_quantile(scores, q):
    """Compute a quantile with the 'higher' interpolation method, compatible across NumPy versions."""
    if len(scores) == 0:
        return 1.0
    q = float(np.clip(q, 0.0, 1.0))
    try:
        return float(np.quantile(scores, q, method='higher'))
    except TypeError:
        return float(np.quantile(scores, q, interpolation='higher'))


def conformal_qhat(scores, alpha):
    """Split-conformal threshold: the ceil((n+1)(1-alpha))-th smallest calibration score.

    FIX H6: when ceil((n+1)(1-alpha)) > n there are too few calibration points for a finite
    threshold; the correct value is +inf (keep the class in every set), not the largest score.
    """
    scores  = np.asarray(scores, dtype=np.float64)
    n       = len(scores)
    if n == 0 or alpha <= 0:
        return float('inf')
    k = int(np.ceil((n + 1) * (1.0 - alpha)))
    if k > n:
        return float('inf')
    return float(np.sort(scores)[k - 1])


def compute_set_metrics(pred_sets, y_true):
    """Compute coverage, set-size statistics, singleton/empty rates from boolean prediction sets."""
    pred_sets = pred_sets.astype(bool)
    set_sizes = pred_sets.sum(axis=1)
    covered   = pred_sets[np.arange(len(y_true)), y_true].astype(int)
    return {
        'empirical_coverage': float(np.mean(covered)),
        'avg_set_size':       float(np.mean(set_sizes)),
        'median_set_size':    float(np.median(set_sizes)),
        'singleton_rate':     float(np.mean(set_sizes == 1)),
        'empty_set_rate':     float(np.mean(set_sizes == 0)),
        'set_sizes':          set_sizes,
        'covered':            covered,
    }


def per_class_coverage_df(pred_sets, y_true, n_classes):
    """Build a DataFrame of per-class empirical coverage and support counts."""
    rows = []
    for c in range(n_classes):
        mask    = (y_true == c)
        support = int(mask.sum())
        cov     = float(np.mean(pred_sets[mask, c])) if support > 0 else np.nan
        rows.append({'class_id': c, 'class_coverage': cov, 'support_count': support})
    return pd.DataFrame(rows)


def normalize_vector(x):
    """Min-max normalise a 1-D array to [0, 1]."""
    x     = np.asarray(x, dtype=np.float64)
    mn, mx = float(np.min(x)), float(np.max(x))
    if abs(mx - mn) < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)


## 7.2 — Plotting Helpers

In [12]:
def fig_to_buffer(fig):
    """Save a matplotlib figure to an in-memory PNG buffer and close the figure."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    return buf


def add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01):
    """Annotate each bar in a bar chart with its numeric value."""
    ymax = ax.get_ylim()[1]
    for p in ax.patches:
        h = p.get_height()
        if np.isnan(h):
            continue
        ax.text(
            p.get_x() + p.get_width() / 2, h + y_pad * ymax,
            fmt.format(h), ha='center', va='bottom', fontsize=10,
        )


def make_per_class_coverage_plot(per_cls_df, alpha, title):
    """Bar chart of per-class empirical coverage with a target coverage reference line."""
    fig, ax = plt.subplots(figsize=(15, 7))
    labels  = [f'Class {int(c)}' for c in per_cls_df['class_id'].tolist()]
    vals    = per_cls_df['class_coverage'].to_numpy(dtype=float)

    ax.bar(labels, vals, edgecolor='black', color='#4C72B0')
    ax.axhline(1 - alpha, color='red', linestyle='--', linewidth=2,
               label=f'Desired Coverage ({1-alpha:.2f})')
    ax.set_title(title, fontsize=16)
    ax.set_xlabel('Class')
    ax.set_ylabel('Achieved Coverage')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.tick_params(axis='x', rotation=45)
    add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01)
    ax.legend(loc='upper right')
    fig.tight_layout()
    return fig_to_buffer(fig)


def make_certain_uncertain_map_plot(set_sizes_map, title):
    """Render a spatial map distinguishing certain (singleton set) from uncertain pixels."""
    # 0 = certain (yellow), 1 = uncertain (dark navy)
    disp = np.where(set_sizes_map == 1, 0, 1)
    cmap = ListedColormap(['#FFFF00', '#001F3F'])

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(disp, cmap=cmap)
    ax.set_title(title, fontsize=16)
    ax.axis('off')

    legend_handles = [
        Patch(facecolor='#FFFF00', edgecolor='black', label='Certain'),
        Patch(facecolor='#001F3F', edgecolor='black', label='Uncertain'),
    ]
    ax.legend(handles=legend_handles, loc='upper left',
              bbox_to_anchor=(1.02, 1), borderaxespad=0.0, frameon=True)
    fig.tight_layout()
    return fig_to_buffer(fig)


def make_class_uncertain_mask_plot(combined_map, n_classes, title):
    """Colour-coded spatial map where uncertain pixels (multi-element prediction sets) are grey."""
    class_colors    = get_class_colors(n_classes)
    uncertain_color = resolve_uncertain_color(class_colors)
    cmap            = ListedColormap(class_colors + [uncertain_color])

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(combined_map, cmap=cmap, vmin=0, vmax=n_classes, interpolation='nearest')
    ax.set_title(title, fontsize=16)
    ax.axis('off')

    cbar = fig.colorbar(im, ax=ax, ticks=np.arange(n_classes + 1), fraction=0.046, pad=0.04)
    cbar.set_ticklabels([f'Class {i}' for i in range(n_classes)] + ['Uncertain'])
    fig.tight_layout()
    return fig_to_buffer(fig)


def make_pixel_counts_plot(pixel_counts_df, title, n_classes):
    """Bar chart of pixel counts per class plus an 'Uncertain' category."""
    class_colors = get_class_colors(n_classes)
    colors       = class_colors + [resolve_uncertain_color(class_colors)]

    fig, ax = plt.subplots(figsize=(12, 6))
    labels  = pixel_counts_df['label'].tolist()
    counts  = pixel_counts_df['pixel_count'].tolist()
    ax.bar(labels, counts, color=colors[:len(labels)], edgecolor='black')
    ax.set_title(title, fontsize=16)
    ax.set_ylabel('Number of Pixels')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.tick_params(axis='x', rotation=45)

    ymax = max(counts) if len(counts) else 1
    for i, v in enumerate(counts):
        ax.text(i, v + 0.01 * ymax, f'{int(v):,}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    fig.tight_layout()
    return fig_to_buffer(fig)


def build_pixel_counts_df(combined_map, n_classes):
    """Tally per-class pixel counts from a combined class/uncertain map."""
    uniq, cnt = np.unique(combined_map, return_counts=True)
    counts    = {int(k): int(v) for k, v in zip(uniq, cnt)}

    rows = []
    for c in range(n_classes):
        rows.append({'class_id': c, 'label': f'Class {c}', 'pixel_count': counts.get(c, 0)})
    rows.append({'class_id': n_classes, 'label': 'Uncertain', 'pixel_count': counts.get(n_classes, 0)})
    return pd.DataFrame(rows)


## 7.3 — DAPM Inference Helpers

`predict_dapm_probs` mirrors `predict_with_dapm_classifier` from the DAPM training notebook
(encoder mean `z_mu` → latent classifier, no diffusion sampling) and layers on the same
`normalize_probs` sanitisation used for the baseline models, so its output is a drop-in
replacement for `predict_probs` everywhere a conformal method expects a probability matrix.
`predict_full_scene_dapm_probs` is the DAPM analogue of `predict_full_scene_probs`: same
column-by-column sliding-window loop, with the DAPM bundle used for inference instead of a
plain `.keras` model.
# DRY: consolidated repeated full-scene inference pattern, parameterised on the probability function

In [13]:
def predict_dapm_probs(bundle, x_data, batch_size=128):
    """Run encoder(z_mu) -> classifier inference through a DAPM bundle; returns sanitised probabilities."""
    feature_extractor = bundle['feature_extractor']
    encoder            = bundle['encoder']
    classifier         = bundle['classifier']

    ds = tf.data.Dataset.from_tensor_slices(x_data.astype(np.float32)).batch(batch_size)
    all_probs = []
    for xb in ds:
        feat       = feature_extractor(xb, training=False)
        z_mu, _, _ = encoder(feat, training=False)
        probs      = classifier(z_mu, training=False)
        all_probs.append(probs.numpy())

    prob = np.concatenate(all_probs, axis=0)
    return normalize_probs(prob, eps=EPS)


def predict_full_scene_dapm_probs(bundle, x_img, H, W, B, patch_size, batch_size=128):
    """Run DAPM patch-based inference over every pixel and return an (H, W, C) probability cube."""
    pad   = patch_size // 2
    x_pad = np.pad(x_img, ((pad, pad), (pad, pad), (0, 0)), mode='edge')

    # Infer number of classes from a single test patch
    test_patch = x_pad[0:patch_size, 0:patch_size, :][None, ...]
    test_prob  = predict_dapm_probs(bundle, test_patch, batch_size=1)
    n_classes  = test_prob.shape[1]

    full_prob = np.zeros((H, W, n_classes), dtype=np.float32)
    for col in range(W):
        patches = np.zeros((H, patch_size, patch_size, B), dtype=np.float32)
        for row in range(H):
            patches[row] = x_pad[row:row + patch_size, col:col + patch_size, :]

        prob_col              = predict_dapm_probs(bundle, patches, batch_size=batch_size)
        full_prob[:, col, :]  = prob_col

        if (col + 1) % 50 == 0 or (col + 1) == W:
            print(f'  full-scene progress: col {col + 1}/{W}')

    assert full_prob.shape == (H, W, n_classes), f'Unexpected full-scene shape: {full_prob.shape}'
    return full_prob


# 8.0 — Conformal Prediction Methods (applied to DAPM probabilities)

Each sub-section implements one conformal method: calibration, evaluation-set scoring,
full-scene map generation, and output packaging (summary dict, per-class DataFrame, plot
buffers, Excel tables). Split CP, Class-Conditional CP, RC3P, and RAPS operate purely on
probability matrices, so they are reused unchanged from the baseline conformal-comparison
notebook — only the *source* of `prob_cal` / `prob_eval` / `prob_full` changes (DAPM
encoder→classifier instead of a plain `.keras` model). Clustered CP is adapted because its
embeddings come from the DAPM latent space rather than a base-model penultimate layer.

## 8.1 — Split Conformal Prediction (SplitCP)

In [14]:
def build_split_outputs_for_model(model_name, y_cal, prob_cal, y_eval, prob_eval, prob_full, alpha=0.05):
    """Run standard Split Conformal Prediction and return a standardised output dict."""
    t0 = time.perf_counter()

    # Calibration
    calib_scores = 1.0 - prob_cal[np.arange(len(y_cal)), y_cal]
    q_hat        = conformal_qhat(calib_scores, alpha)

    # Evaluation
    pred_sets_eval = prob_eval >= (1.0 - q_hat)
    metrics  = compute_set_metrics(pred_sets_eval, y_eval)
    per_cls  = per_class_coverage_df(pred_sets_eval, y_eval, prob_eval.shape[1])

    # Full-scene maps
    pred_sets_full  = prob_full >= (1.0 - q_hat)
    set_sizes_map   = np.sum(pred_sets_full, axis=2)
    pred_class_map  = np.argmax(prob_full, axis=2)
    combined_map    = np.where(set_sizes_map == 1, pred_class_map, prob_full.shape[2])
    pixel_counts_df = build_pixel_counts_df(combined_map, prob_full.shape[2])

    plot_buffers = {
        'Per-Class Coverage': make_per_class_coverage_plot(
            per_cls, alpha=alpha,
            title='Standard Split Conformal Prediction: Per-Class Coverage',
        ),
        'Certain vs Uncertain Map': make_certain_uncertain_map_plot(
            set_sizes_map,
            title='Predictions with 95% Uncertainty Map\n(Split Conformal Prediction — SCP)',
        ),
        'Class Map with Uncertain Mask': make_class_uncertain_mask_plot(
            combined_map, n_classes=prob_full.shape[2],
            title='Predictions with 95% Uncertainty Mask\n(Split Conformal Prediction — SCP)',
        ),
        'Pixel Counts': make_pixel_counts_plot(
            pixel_counts_df,
            title='Pixel Count per Class (Including Uncertain Regions)',
            n_classes=prob_full.shape[2],
        ),
    }

    runtime = time.perf_counter() - t0
    summary = {
        'model_name': model_name, 'method': 'SplitConformal',
        'target_coverage': float(1.0 - alpha),
        'empirical_coverage': metrics['empirical_coverage'],
        'avg_set_size': metrics['avg_set_size'],
        'median_set_size': metrics['median_set_size'],
        'singleton_rate': metrics['singleton_rate'],
        'empty_set_rate': metrics['empty_set_rate'],
        'runtime_sec': float(runtime),
        'alpha': float(alpha), 'lam': np.nan, 'n_clusters': np.nan,
        'mean_per_class_coverage': float(per_cls['class_coverage'].mean(skipna=True)),
    }

    tables = {
        'Summary':                 pd.DataFrame([summary]),
        'Per-Class Coverage Values': per_cls,
        'Pixel Counts':            pixel_counts_df,
        'Threshold':               pd.DataFrame([{'q_hat_split': float(q_hat)}]),
    }

    return {
        'model_name': model_name, 'method': 'SplitConformal',
        'summary': summary, 'per_class_df': per_cls,
        'plot_buffers': plot_buffers, 'tables': tables,
    }


## 8.2 — Class-Conditional Conformal Prediction (CcCP)

In [15]:
def build_classconditional_outputs_for_model(model_name, y_cal, prob_cal, y_eval, prob_eval, prob_full, alpha=0.05):
    """Run Class-Conditional Conformal Prediction with per-class q_hat thresholds."""
    t0 = time.perf_counter()
    n_classes = prob_cal.shape[1]

    # Per-class calibration thresholds
    q_hats = np.zeros(n_classes, dtype=np.float64)
    for c in range(n_classes):
        mask = (y_cal == c)
        if mask.sum() == 0:
            q_hats[c] = 1.0
            continue
        scores_c  = 1.0 - prob_cal[mask, c]
        q_hats[c] = conformal_qhat(scores_c, alpha)

    # Evaluation
    thresholds     = 1.0 - q_hats.reshape(1, -1)
    pred_sets_eval = prob_eval >= thresholds
    metrics  = compute_set_metrics(pred_sets_eval, y_eval)
    per_cls  = per_class_coverage_df(pred_sets_eval, y_eval, n_classes)

    # Full-scene maps
    pred_sets_full  = prob_full >= (1.0 - q_hats.reshape(1, 1, -1))
    set_sizes_map   = np.sum(pred_sets_full, axis=2)
    pred_class_map  = np.argmax(prob_full, axis=2)
    combined_map    = np.where(set_sizes_map == 1, pred_class_map, n_classes)
    pixel_counts_df = build_pixel_counts_df(combined_map, n_classes)

    plot_buffers = {
        'Per-Class Coverage': make_per_class_coverage_plot(
            per_cls, alpha=alpha,
            title='Class-Conditional Conformal Prediction: Per-Class Coverage',
        ),
        'Certain vs Uncertain Map': make_certain_uncertain_map_plot(
            set_sizes_map,
            title='Predictions with 95% Uncertainty Map\n(Class-Conditional Conformal Prediction — CcCP)',
        ),
        'Class Map with Uncertain Mask': make_class_uncertain_mask_plot(
            combined_map, n_classes=n_classes,
            title='Predictions with 95% Uncertainty Mask\n(Class-Conditional Conformal Prediction — CcCP)',
        ),
        'Pixel Counts': make_pixel_counts_plot(
            pixel_counts_df,
            title='Pixel Count per Class (Including Uncertain Regions)',
            n_classes=n_classes,
        ),
    }

    runtime = time.perf_counter() - t0
    summary = {
        'model_name': model_name, 'method': 'ClassConditionalConformal',
        'target_coverage': float(1.0 - alpha),
        'empirical_coverage': metrics['empirical_coverage'],
        'avg_set_size': metrics['avg_set_size'],
        'median_set_size': metrics['median_set_size'],
        'singleton_rate': metrics['singleton_rate'],
        'empty_set_rate': metrics['empty_set_rate'],
        'runtime_sec': float(runtime),
        'alpha': float(alpha), 'lam': np.nan, 'n_clusters': np.nan,
        'mean_per_class_coverage': float(per_cls['class_coverage'].mean(skipna=True)),
    }
    qhat_df = pd.DataFrame({'class_id': np.arange(n_classes), 'q_hat_classconditional': q_hats})

    tables = {
        'Summary':                 pd.DataFrame([summary]),
        'Per-Class Coverage Values': per_cls,
        'Pixel Counts':            pixel_counts_df,
        'Classwise q_hat':         qhat_df,
    }

    return {
        'model_name': model_name, 'method': 'ClassConditionalConformal',
        'summary': summary, 'per_class_df': per_cls,
        'plot_buffers': plot_buffers, 'tables': tables,
    }


## 8.3 — Rank Calibrated Class-Conditional CP (RC3P)

In [16]:
def compute_topk_accuracy_matrix(prob, y, n_classes):
    """Compute a (K x C) matrix where entry [k, c] is top-k accuracy for class c."""
    acc_matrix = np.zeros((n_classes, n_classes))
    # Double argsort of negative probs gives 1-based rank (1 = highest prob)
    ranks = np.argsort(np.argsort(-prob, axis=1), axis=1) + 1
    for c in range(n_classes):
        mask        = (y == c)
        if mask.sum() == 0:
            continue
        class_ranks = ranks[mask, c]
        for k in range(1, n_classes + 1):
            acc_matrix[k-1, c] = np.mean(class_ranks <= k)
    return acc_matrix


def compute_rc3p_qhats_and_sets(prob_cal, y_cal, prob_eval, alpha, truncated_gap=0.1):
    """RC3P search: find optimal truncated rank limits and per-class q_hats minimising average set size."""
    n_classes = prob_cal.shape[1]
    num_cal   = len(y_cal)

    cal_ranks  = np.argsort(np.argsort(-prob_cal,  axis=1), axis=1) + 1
    eval_ranks = np.argsort(np.argsort(-prob_eval, axis=1), axis=1) + 1

    acc_matrix = compute_topk_accuracy_matrix(prob_cal, y_cal, n_classes)
    err_matrix = 1.0 - acc_matrix

    num_samples_per_class = num_cal / n_classes
    tc_alpha = alpha - (truncated_gap / np.sqrt(num_samples_per_class))

    # Minimum rank k such that top-k error falls below tc_alpha
    suit_k = []
    for c in range(n_classes):
        valid_k = np.where(err_matrix[:, c] < tc_alpha)[0]
        suit_k.append(valid_k[0] + 1 if len(valid_k) > 0 else n_classes)

    k_max    = max(suit_k)
    rank_all = n_classes
    mix_paras = np.linspace(0, 1, int(rank_all - k_max) + 1) if rank_all >= k_max else [1.0]

    smallest_ps          = float('inf')
    best_classwise_qhats = np.ones(n_classes)
    best_suit_indices    = suit_k

    for mix_para in mix_paras:
        test_indices = [int(np.ceil((1 - mix_para) * suit_k[i] + n_classes * mix_para))
                        for i in range(n_classes)]
        test_err     = [err_matrix[test_indices[i]-1, i] for i in range(n_classes)]
        test_alphas  = [tc_alpha - err for err in test_err]

        q_hats = np.zeros(n_classes)
        for c in range(n_classes):
            idx    = (y_cal == c) & (cal_ranks[:, c] <= test_indices[c])
            scores = 1.0 - prob_cal[idx, c]
            q_hats[c] = 1.0 if len(scores) == 0 else conformal_qhat(scores, test_alphas[c])

        thresholds   = 1.0 - q_hats
        meets_thresh = prob_eval >= thresholds
        meets_rank   = eval_ranks <= np.array(test_indices)
        pred_sets    = meets_thresh & meets_rank

        avg_size = np.mean(pred_sets.sum(axis=1))
        if avg_size < smallest_ps:
            smallest_ps          = avg_size
            best_classwise_qhats = q_hats
            best_suit_indices    = test_indices

    return best_classwise_qhats, best_suit_indices


def build_rc3p_outputs_for_model(model_name, y_cal, prob_cal, y_eval, prob_eval, prob_full, alpha=0.05, truncated_gap=0.1):
    """Run RC3P and return a standardised output dict including full-scene maps."""
    t0        = time.perf_counter()
    n_classes = prob_cal.shape[1]

    # FIX H8: the set-size criterion is evaluated on calibration data (was: on the eval set)
    q_hats, suit_indices = compute_rc3p_qhats_and_sets(
        prob_cal, y_cal, prob_cal, alpha, truncated_gap,
    )

    # Evaluation
    eval_ranks       = np.argsort(np.argsort(-prob_eval, axis=1), axis=1) + 1
    meets_thresh_eval = prob_eval >= (1.0 - q_hats.reshape(1, -1))
    meets_rank_eval   = eval_ranks <= np.array(suit_indices).reshape(1, -1)
    pred_sets_eval    = meets_thresh_eval & meets_rank_eval

    metrics = compute_set_metrics(pred_sets_eval, y_eval)
    per_cls = per_class_coverage_df(pred_sets_eval, y_eval, n_classes)

    # Full-scene maps
    full_ranks        = np.argsort(np.argsort(-prob_full, axis=2), axis=2) + 1
    meets_thresh_full = prob_full >= (1.0 - q_hats.reshape(1, 1, -1))
    meets_rank_full   = full_ranks <= np.array(suit_indices).reshape(1, 1, -1)
    pred_sets_full    = meets_thresh_full & meets_rank_full

    set_sizes_map   = np.sum(pred_sets_full, axis=2)
    pred_class_map  = np.argmax(prob_full, axis=2)
    combined_map    = np.where(set_sizes_map == 1, pred_class_map, n_classes)
    pixel_counts_df = build_pixel_counts_df(combined_map, n_classes)

    plot_buffers = {
        'Per-Class Coverage': make_per_class_coverage_plot(
            per_cls, alpha=alpha, title='RC3P: Per-Class Coverage',
        ),
        'Certain vs Uncertain Map': make_certain_uncertain_map_plot(
            set_sizes_map, title='Predictions with 95% Uncertainty Map\n(RC3P)',
        ),
        'Class Map with Uncertain Mask': make_class_uncertain_mask_plot(
            combined_map, n_classes=n_classes,
            title='Predictions with 95% Uncertainty Mask\n(RC3P)',
        ),
        'Pixel Counts': make_pixel_counts_plot(
            pixel_counts_df, title='Pixel Count per Class (RC3P)', n_classes=n_classes,
        ),
    }

    runtime = time.perf_counter() - t0
    summary = {
        'model_name': model_name, 'method': 'RC3P',
        'target_coverage': float(1.0 - alpha),
        'empirical_coverage': metrics['empirical_coverage'],
        'avg_set_size': metrics['avg_set_size'],
        'median_set_size': metrics['median_set_size'],
        'singleton_rate': metrics['singleton_rate'],
        'empty_set_rate': metrics['empty_set_rate'],
        'runtime_sec': float(runtime),
        'alpha': float(alpha), 'lam': np.nan, 'n_clusters': np.nan,
        'mean_per_class_coverage': float(per_cls['class_coverage'].mean(skipna=True)),
    }
    qhat_df = pd.DataFrame({
        'class_id': np.arange(n_classes),
        'q_hat_rc3p': q_hats,
        'suit_index_limit': suit_indices,
    })

    tables = {
        'Summary':                  pd.DataFrame([summary]),
        'Per-Class Coverage Values': per_cls,
        'Pixel Counts':             pixel_counts_df,
        'RC3P Thresholds & Limits': qhat_df,
    }

    return {
        'model_name': model_name, 'method': 'RC3P',
        'summary': summary, 'per_class_df': per_cls,
        'plot_buffers': plot_buffers, 'tables': tables,
    }


## 8.4 — Clustered Conformal Prediction (ClCP) — DAPM latent-space embeddings

Adapted from `build_clustered_outputs_for_model`: instead of slicing the penultimate layer of
a plain `.keras` model, embeddings are the DAPM encoder's mean latent vector `z_mu` — the same
representation the DAPM classifier itself operates on, obtained via the bundle's
`feature_extractor` -> `encoder` path. Every other step (class-mean clustering, per-cluster
calibration, coverage metrics, plots) is unchanged.
# DRY: same clustering / calibration logic as the baseline version, parameterised on the embedding source

In [17]:
def get_dapm_embeddings(bundle, x_data, batch_size=128):
    """Return DAPM latent-space embeddings (encoder mean z_mu) for a patch array."""
    feature_extractor = bundle['feature_extractor']
    encoder            = bundle['encoder']

    ds = tf.data.Dataset.from_tensor_slices(x_data.astype(np.float32)).batch(batch_size)
    all_emb = []
    for xb in ds:
        feat       = feature_extractor(xb, training=False)
        z_mu, _, _ = encoder(feat, training=False)
        all_emb.append(z_mu.numpy())
    emb = np.concatenate(all_emb, axis=0)
    return np.nan_to_num(emb)


def build_clustered_outputs_for_model_dapm(
    model_name, bundle, x_cal, y_cal, x_eval, y_eval,
    prob_cal, prob_eval, alpha=0.05, n_clusters=4, batch_size=128,
):
    """Run Clustered Conformal Prediction on DAPM latent embeddings, cluster classes, calibrate per cluster."""
    t0 = time.perf_counter()
    n_classes = prob_cal.shape[1]
    k = int(min(max(1, n_clusters), n_classes))

    # Build embeddings from the DAPM latent space (z_mu)
    emb_cal = get_dapm_embeddings(bundle, x_cal, batch_size=batch_size)

    # FIX M19: fit clusters on one half of the calibration data and compute the
    # conformal quantiles on the other half (Ding et al., 2023)
    _perm   = np.random.default_rng(SEED).permutation(len(y_cal))
    _clu    = np.zeros(len(y_cal), dtype=bool); _clu[_perm[: len(y_cal) // 2]] = True
    global_mean = emb_cal[_clu].mean(axis=0)
    class_means = []
    for c in range(n_classes):
        m = (y_cal == c) & _clu
        class_means.append(emb_cal[m].mean(axis=0) if m.sum() > 0 else global_mean)
    class_means = np.vstack(class_means)

    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    cluster_assignments = km.fit_predict(class_means)

    # Per-cluster calibration
    q_hats_per_cluster = np.ones(k, dtype=np.float64)
    for cluster_id in range(k):
        cls  = np.where(cluster_assignments == cluster_id)[0]
        mask = np.isin(y_cal, cls) & ~_clu
        if mask.sum() == 0:
            continue
        scores = 1.0 - prob_cal[mask, y_cal[mask]]
        q_hats_per_cluster[cluster_id] = conformal_qhat(scores, alpha)

    q_per_class = q_hats_per_cluster[cluster_assignments]
    pred_sets   = prob_eval >= (1.0 - q_per_class.reshape(1, -1))

    metrics = compute_set_metrics(pred_sets, y_eval)
    per_cls = per_class_coverage_df(pred_sets, y_eval, n_classes)

    # Uncertainty analysis
    set_sizes       = pred_sets.sum(axis=1)
    norm_unc        = normalize_vector(set_sizes)
    pred_class      = np.argmax(prob_eval, axis=1)
    sample_clusters = cluster_assignments[pred_class]

    cluster_mean_unc_df = (
        pd.DataFrame({'cluster': sample_clusters, 'uncertainty': norm_unc})
        .groupby('cluster', as_index=False)['uncertainty'].mean()
        .sort_values('cluster').reset_index(drop=True)
    )

    class_rows = []
    for c in range(n_classes):
        m = (pred_class == c)
        if m.sum() == 0:
            continue
        class_rows.append({'class_id': c, 'uncertainty': float(np.mean(norm_unc[m]))})
    class_mean_unc_df = pd.DataFrame(class_rows)

    # Plots
    plot_buffers = {}
    plot_buffers['Per-Class Coverage'] = make_per_class_coverage_plot(
        per_cls, alpha=alpha,
        title='Clustered Conformal Prediction (DAPM latent space) — Per-Class Coverage',
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(norm_unc, bins=30, color='steelblue', edgecolor='black')
    ax.set_xlabel('Normalized Predictive Set Size (Uncertainty)')
    ax.set_ylabel('Number of Samples')
    ax.set_title('Uncertainty Distribution — Clustered Conformal Prediction (DAPM ClCP)')
    ax.grid(axis='y', alpha=0.4)
    fig.tight_layout()
    plot_buffers['Uncertainty Distribution'] = fig_to_buffer(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(cluster_mean_unc_df['cluster'].astype(str), cluster_mean_unc_df['uncertainty'],
           color='darkorange', edgecolor='black')
    ax.set_title('Average Uncertainty per Cluster (DAPM ClCP)')
    ax.set_ylabel('Mean Normalized Uncertainty')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01)
    fig.tight_layout()
    plot_buffers['Avg Uncertainty per Cluster'] = fig_to_buffer(fig)

    fig, ax = plt.subplots(figsize=(12, 5))
    if len(class_mean_unc_df) > 0:
        ax.bar(class_mean_unc_df['class_id'].astype(str), class_mean_unc_df['uncertainty'],
               color='#1f77b4', edgecolor='black')
        ax.set_xticklabels([f'Class {x}' for x in class_mean_unc_df['class_id']], rotation=45, ha='right')
    ax.set_xlabel('Class')
    ax.set_ylabel('Mean Normalized Uncertainty')
    ax.set_title('Average Uncertainty per Class — Clustered Conformal Prediction (DAPM ClCP)')
    ax.grid(axis='y', alpha=0.4)
    add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01)
    fig.tight_layout()
    plot_buffers['Avg Uncertainty per Class'] = fig_to_buffer(fig)

    runtime = time.perf_counter() - t0
    summary = {
        'model_name': model_name, 'method': 'ClusteredConformal',
        'target_coverage': float(1.0 - alpha),
        'empirical_coverage': metrics['empirical_coverage'],
        'avg_set_size': metrics['avg_set_size'],
        'median_set_size': metrics['median_set_size'],
        'singleton_rate': metrics['singleton_rate'],
        'empty_set_rate': metrics['empty_set_rate'],
        'runtime_sec': float(runtime),
        'alpha': float(alpha), 'lam': np.nan, 'n_clusters': int(k),
        'mean_per_class_coverage': float(per_cls['class_coverage'].mean(skipna=True)),
    }

    qhat_cluster_df  = pd.DataFrame({'cluster_id': np.arange(k), 'q_hat_cluster': q_hats_per_cluster})
    class_cluster_df = pd.DataFrame({'class_id': np.arange(n_classes), 'cluster_id': cluster_assignments})

    tables = {
        'Summary':                    pd.DataFrame([summary]),
        'Per-Class Coverage Values':  per_cls,
        'Class-to-Cluster Assignment': class_cluster_df,
        'Cluster q_hat':              qhat_cluster_df,
        'Cluster Mean Uncertainty':   cluster_mean_unc_df,
        'Class Mean Uncertainty':     class_mean_unc_df,
    }

    return {
        'model_name': model_name, 'method': 'ClusteredConformal',
        'summary': summary, 'per_class_df': per_cls,
        'plot_buffers': plot_buffers, 'tables': tables,
    }


## 8.5 — RAPS (Regularised Adaptive Prediction Sets)

In [18]:
def raps_score_single(prob_row, true_label, lam=0.01, k_reg=1, u=None):
    """Randomised RAPS score (Angelopoulos et al., 2021).

    FIX H7: E = sum of probabilities ranked above y  +  u * p_y  +  lam * (rank_y - k_reg)^+,
    with rank_y 1-based and u ~ U(0,1). The old score left out p_y and had no u, so q_hat
    collapsed to 0 and every set became top-1.
    """
    order = np.argsort(prob_row)[::-1]
    rank0 = int(np.where(order == true_label)[0][0])          # 0-based position of y
    u     = np.random.rand() if u is None else u
    above = float(np.sum(prob_row[order[:rank0]]))
    return above + u * float(prob_row[true_label]) + float(lam) * max(rank0 + 1 - int(k_reg), 0)


def raps_set_single(prob_row, q_hat, lam=0.01, k_reg=1, u=None):
    """RAPS prediction set {y : E(x, y, u) <= q_hat} using the same u for every candidate class."""
    order    = np.argsort(prob_row)[::-1]
    p_sorted = prob_row[order]
    u        = np.random.rand() if u is None else u
    above    = np.concatenate([[0.0], np.cumsum(p_sorted)[:-1]])
    ranks    = np.arange(1, len(prob_row) + 1)
    scores   = above + u * p_sorted + float(lam) * np.maximum(ranks - int(k_reg), 0)
    pred_set = np.zeros_like(prob_row, dtype=bool)
    pred_set[order[scores <= q_hat]] = True
    return pred_set


def build_raps_outputs_for_model(model_name, y_cal, prob_cal, y_eval, prob_eval, alpha=0.05, lam=0.01, k_reg=1):
    """Run RAPS and return a standardised output dict (no full-scene map for RAPS)."""
    t0    = time.perf_counter()
    n_cal = len(y_cal)

    raps_scores = np.array([
        raps_score_single(prob_cal[i], int(y_cal[i]), lam=lam, k_reg=k_reg)
        for i in range(n_cal)
    ], dtype=np.float64)

    q_hat = conformal_qhat(raps_scores, alpha)

    pred_sets = np.array([
        raps_set_single(prob_eval[i], q_hat=q_hat, lam=lam, k_reg=k_reg)
        for i in range(len(y_eval))
    ], dtype=bool)

    metrics = compute_set_metrics(pred_sets, y_eval)
    per_cls = per_class_coverage_df(pred_sets, y_eval, prob_eval.shape[1])

    plot_buffers = {
        'Per-Class Coverage': make_per_class_coverage_plot(
            per_cls, alpha=alpha, title='RAPS: Per-Class Coverage',
        ),
    }

    runtime = time.perf_counter() - t0
    summary = {
        'model_name': model_name, 'method': 'RAPS',
        'target_coverage': float(1.0 - alpha),
        'empirical_coverage': metrics['empirical_coverage'],
        'avg_set_size': metrics['avg_set_size'],
        'median_set_size': metrics['median_set_size'],
        'singleton_rate': metrics['singleton_rate'],
        'empty_set_rate': metrics['empty_set_rate'],
        'runtime_sec': float(runtime),
        'alpha': float(alpha), 'lam': float(lam), 'n_clusters': np.nan,
        'mean_per_class_coverage': float(per_cls['class_coverage'].mean(skipna=True)),
    }

    tables = {
        'Summary':                 pd.DataFrame([summary]),
        'Per-Class Coverage Values': per_cls,
        'RAPS Parameters':         pd.DataFrame([{'q_hat_raps': float(q_hat), 'lambda': float(lam), 'k_reg': int(k_reg)}]),
    }

    return {
        'model_name': model_name, 'method': 'RAPS',
        'summary': summary, 'per_class_df': per_cls,
        'plot_buffers': plot_buffers, 'tables': tables,
    }


# 9.0 — Main Execution Loop

Iterates over all three backbones. For each: loads its DAPM bundle, computes calibration /
evaluation / full-scene probabilities via the encoder→classifier path (Section 7.3), then
runs each of the five conformal prediction methods against those DAPM probabilities and
collects the results into `all_outputs`.

In [19]:
def method_sheet_prefix(method_name):
    """Return the abbreviated sheet-name prefix for a given method identifier."""
    mapping = {
        'SplitConformal':          'Split',
        'ClassConditionalConformal': 'ClassCond',
        'RC3P':                    'RC3P',
        'ClusteredConformal':      'Clustered',
        'RAPS':                    'RAPS',
    }
    return mapping.get(method_name, method_name)


all_outputs           = []
full_scene_prob_cache = {}
dapm_bundles           = {}

for model_key in MODEL_KEYS:
    model_name = MODEL_NAME_MAP.get(model_key, model_key)
    print(f'\n==================== {model_name} ====================')

    bundle = load_dapm_bundle(model_key)
    dapm_bundles[model_key] = bundle

    prob_cal  = predict_dapm_probs(bundle, x_cal,  batch_size=BATCH_SIZE)
    prob_eval = predict_dapm_probs(bundle, x_eval, batch_size=BATCH_SIZE)

    # Cache full-scene probabilities (expensive) so they are computed only once per model
    if model_key not in full_scene_prob_cache:
        print(f'Computing full-scene DAPM probabilities for {model_name} ...')
        full_scene_prob_cache[model_key] = predict_full_scene_dapm_probs(
            bundle=bundle, x_img=x_img, H=H, W=W, B=B,
            patch_size=PATCH_SIZE, batch_size=BATCH_SIZE,
        )
    prob_full = full_scene_prob_cache[model_key]

    # 1. Split CP
    split_out = build_split_outputs_for_model(
        model_name=model_name, y_cal=y_cal, prob_cal=prob_cal,
        y_eval=y_eval, prob_eval=prob_eval, prob_full=prob_full, alpha=ALPHA,
    )
    all_outputs.append(split_out)

    # 2. Class-Conditional CP
    cw_out = build_classconditional_outputs_for_model(
        model_name=model_name, y_cal=y_cal, prob_cal=prob_cal,
        y_eval=y_eval, prob_eval=prob_eval, prob_full=prob_full, alpha=ALPHA,
    )
    all_outputs.append(cw_out)

    # 3. RC3P
    rc3p_out = build_rc3p_outputs_for_model(
        model_name=model_name, y_cal=y_cal, prob_cal=prob_cal,
        y_eval=y_eval, prob_eval=prob_eval, prob_full=prob_full,
        alpha=ALPHA, truncated_gap=0.1,
    )
    all_outputs.append(rc3p_out)

    # 4. Clustered CP (DAPM latent-space embeddings)
    clcp_out = build_clustered_outputs_for_model_dapm(
        model_name=model_name, bundle=bundle, x_cal=x_cal, y_cal=y_cal,
        x_eval=x_eval, y_eval=y_eval, prob_cal=prob_cal, prob_eval=prob_eval,
        alpha=ALPHA, n_clusters=N_CLUSTERS, batch_size=BATCH_SIZE,
    )
    all_outputs.append(clcp_out)

    # 5. RAPS
    raps_out = build_raps_outputs_for_model(
        model_name=model_name, y_cal=y_cal, prob_cal=prob_cal,
        y_eval=y_eval, prob_eval=prob_eval, alpha=ALPHA, lam=RAPS_LAM, k_reg=RAPS_K_REG,
    )
    all_outputs.append(raps_out)

    for out in [split_out, cw_out, rc3p_out, clcp_out, raps_out]:
        s = out['summary']
        print(f"{s['method']}: coverage={s['empirical_coverage']:.4f}, "
              f"avg_set={s['avg_set_size']:.3f}, runtime={s['runtime_sec']:.2f}s")

    # Free model and intermediate probability arrays from memory
    del bundle, prob_cal, prob_eval
    import gc
    gc.collect()


# Aggregate summary tables
summary_compact_df = (
    pd.DataFrame([o['summary'] for o in all_outputs])
    .sort_values(['method', 'model_name'])
    .reset_index(drop=True)
)

per_class_all_df = pd.concat([
    o['per_class_df'].assign(model_name=o['model_name'], method=o['method'])
    for o in all_outputs
], ignore_index=True)



==================== AlexNet ====================
  Loading base model from: /kaggle/working/models/AlexNet_CNN_best.keras


  Weights loaded for AlexNet_CNN


Computing full-scene DAPM probabilities for AlexNet ...


  full-scene progress: col 50/145


  full-scene progress: col 100/145


  full-scene progress: col 145/145


... (9 outputs left out)

  full-scene progress: col 50/145


  full-scene progress: col 100/145


  full-scene progress: col 145/145


SplitConformal: coverage=0.9769, avg_set=9.037, runtime=1.71s
ClassConditionalConformal: coverage=0.9523, avg_set=9.393, runtime=1.70s
RC3P: coverage=0.9567, avg_set=8.405, runtime=1.70s
ClusteredConformal: coverage=0.9776, avg_set=10.934, runtime=4.62s
RAPS: coverage=0.9798, avg_set=10.257, runtime=0.48s


# 10.0 — Comparative Analysis Plots

Generates grouped bar charts comparing empirical coverage, average set size, runtime, and
mean per-class coverage across all three DAPM-backed models — separately for Clustered CP
and RAPS.

In [20]:
def build_method_comparison_buffers(summary_df, method_name, title_prefix):
    """Build grouped metric and mean per-class coverage plots for a given method across all models."""
    df = summary_df[summary_df['method'] == method_name].copy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    sns.barplot(data=df, x='model_name', y='empirical_coverage', ax=axes[0], palette='Set2')
    axes[0].axhline(1 - ALPHA, linestyle='--', color='red', linewidth=1.8,
                    label=f'Target ({1-ALPHA:.2f})')
    axes[0].set_title(f'{title_prefix}: Empirical Coverage')
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('Coverage')
    axes[0].set_ylim(0, 1.1)
    axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=True)

    sns.barplot(data=df, x='model_name', y='avg_set_size',  ax=axes[1], palette='Set3')
    axes[1].set_title(f'{title_prefix}: Avg Set Size')
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('Average Set Size')

    sns.barplot(data=df, x='model_name', y='runtime_sec',   ax=axes[2], palette='Set1')
    axes[2].set_title(f'{title_prefix}: Runtime')
    axes[2].set_xlabel('Model')
    axes[2].set_ylabel('Runtime (sec)')

    for ax in axes:
        add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01)
        ax.grid(axis='y', linestyle='--', alpha=0.4)

    fig.tight_layout()
    metric_buf = fig_to_buffer(fig)

    mean_pc = (
        per_class_all_df[per_class_all_df['method'] == method_name]
        .groupby('model_name', as_index=False)['class_coverage']
        .mean()
        .rename(columns={'class_coverage': 'mean_per_class_coverage'})
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=mean_pc, x='model_name', y='mean_per_class_coverage', palette='Dark2', ax=ax)
    ax.set_title(f'{title_prefix}: Mean Per-Class Coverage')
    ax.set_xlabel('Model')
    ax.set_ylabel('Mean Coverage')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01)
    fig.tight_layout()
    mean_pc_buf = fig_to_buffer(fig)

    return {
        'metrics_plot':       metric_buf,
        'mean_per_class_plot': mean_pc_buf,
        'method_df':          df,
        'mean_per_class_df':  mean_pc,
    }


cluster_compare = build_method_comparison_buffers(summary_compact_df, 'ClusteredConformal', 'Clustered CP')
raps_compare    = build_method_comparison_buffers(summary_compact_df, 'RAPS', 'RAPS')


# 11.0 — Excel Export

Writes all results to a single `.xlsx` workbook. Each method+model combination gets its own
sheet; global summary and comparison sheets are added on top.

## 11.1 — Excel I/O Helpers

In [21]:
def sanitize_sheet_name(name):
    """Strip characters that are illegal in Excel sheet names and truncate to 31 chars."""
    for ch in ['\\', '/', '*', '?', ':', '[', ']']:
        name = name.replace(ch, '_')
    return name[:31]


def make_sheet_name(base, used):
    """Return a unique sheet name derived from base, appending a counter if necessary."""
    base      = sanitize_sheet_name(base)
    candidate = base
    i = 1
    while candidate in used:
        suffix    = f'_{i}'
        candidate = base[:31 - len(suffix)] + suffix
        i += 1
    used.add(candidate)
    return candidate


def insert_buffer_image(ws, row, col, img_buf, x_scale=0.8, y_scale=0.8):
    """Insert a PNG buffer image at (row, col) in an xlsxwriter worksheet."""
    img_buf.seek(0)
    ws.insert_image(row, col, 'plot.png', {'image_data': img_buf, 'x_scale': x_scale, 'y_scale': y_scale})


def write_method_model_sheet(writer, workbook, output, sheet_name):
    """Write tables on the left and plot images on the right for a single method+model sheet."""
    ws = workbook.add_worksheet(sheet_name)
    writer.sheets[sheet_name] = ws

    row = 0
    ws.write(row, 0, f"{output['method']} - {output['model_name']}")
    row += 2

    for tname, tdf in output['tables'].items():
        ws.write(row, 0, tname)
        tdf.to_excel(writer, sheet_name=sheet_name, startrow=row + 1, startcol=0, index=False)
        row += len(tdf) + 4

    # Stacked images to the right so they do not overlap the tables
    img_row, img_col = 0, 9
    for pname, pbuf in output['plot_buffers'].items():
        ws.write(img_row, img_col, pname)
        insert_buffer_image(ws, img_row + 1, img_col, pbuf, x_scale=0.75, y_scale=0.75)
        img_row += 24


def write_comparison_sheet(writer, workbook, sheet_name, compare_obj, title):
    """Write a comparison sheet with summary tables and grouped metric/per-class coverage plots."""
    ws = workbook.add_worksheet(sheet_name)
    writer.sheets[sheet_name] = ws

    ws.write(0, 0, title)
    compare_obj['method_df'].to_excel(writer, sheet_name=sheet_name, startrow=2, startcol=0, index=False)
    compare_obj['mean_per_class_df'].to_excel(writer, sheet_name=sheet_name, startrow=2, startcol=8, index=False)

    ws.write(12, 0, 'Grouped Metrics Comparison')
    insert_buffer_image(ws, 13, 0, compare_obj['metrics_plot'], x_scale=0.78, y_scale=0.78)

    ws.write(40, 0, 'Mean Per-Class Coverage Comparison')
    insert_buffer_image(ws, 41, 0, compare_obj['mean_per_class_plot'], x_scale=0.85, y_scale=0.85)


## 11.2 — Save CSVs, Run Config, and Workbook

In [22]:
run_config = {
    'seed': SEED, 'data_file': str(DATA_FILE), 'label_file': str(LABEL_FILE),
    'model_dir': str(MODEL_DIR), 'dapm_dir': str(DAPM_DIR),
    'output_dir': str(OUTPUT_DIR), 'results_dir': str(RESULTS_DIR),
    'h': H, 'w': W, 'b': B, 'patch_size': PATCH_SIZE,
    'train_percent': TRAIN_PERCENT, 'calib_fraction_of_test': CALIB_FRACTION_OF_TEST,
    'alpha': ALPHA, 'raps_lam': RAPS_LAM, 'raps_k_reg': RAPS_K_REG,
    'n_clusters': N_CLUSTERS, 'batch_size': BATCH_SIZE,
    'probability_source': 'DAPM encoder(z_mu) -> classifier (no diffusion sampling)',
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
}

# CSV / JSON side-outputs
summary_compact_df.to_csv(SUMMARY_CSV_PATH, index=False)
per_class_all_df.to_csv(PER_CLASS_CSV_PATH, index=False)
with open(RUN_CONFIG_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(run_config, f, indent=2)

# Excel workbook
used = set()
try:
    with pd.ExcelWriter(EXCEL_PATH, engine='xlsxwriter', engine_kwargs={'options': {'nan_inf_to_errors': True}}) as writer:
        workbook = writer.book

        s_summary = make_sheet_name('Summary_Compact', used)
        s_config  = make_sheet_name('Run_Config', used)
        summary_compact_df.to_excel(writer, sheet_name=s_summary, index=False)
        pd.DataFrame(list(run_config.items()), columns=['key', 'value']).to_excel(
            writer, sheet_name=s_config, index=False,
        )

        for out in all_outputs:
            method_prefix = method_sheet_prefix(out['method'])
            sheet = make_sheet_name(f"{method_prefix}_{out['model_name']}", used)
            write_method_model_sheet(writer, workbook, out, sheet)

        s_cmp_cluster = make_sheet_name('Compare_Clustered', used)
        s_cmp_raps    = make_sheet_name('Compare_RAPS',      used)
        write_comparison_sheet(writer, workbook, s_cmp_cluster, cluster_compare, 'Clustered CP (DAPM): 3-Model Comparison')
        write_comparison_sheet(writer, workbook, s_cmp_raps,    raps_compare,    'RAPS (DAPM): 3-Model Comparison')
except Exception as _xl_ex:
    print(f'Notice: xlsxwriter Excel export fallback to openpyxl ({_xl_ex})')
    with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
        summary_compact_df.to_excel(writer, sheet_name='Summary_Compact', index=False)
        pd.DataFrame(list(run_config.items()), columns=['key', 'value']).to_excel(
            writer, sheet_name='Run_Config', index=False,
        )
        per_class_all_df.to_excel(writer, sheet_name='PerClass_All', index=False)
        if not cluster_compare.empty:
            cluster_compare.to_excel(writer, sheet_name='Compare_Clustered', index=False)
        if not raps_compare.empty:
            raps_compare.to_excel(writer, sheet_name='Compare_RAPS', index=False)

print('Saved workbook:     ', EXCEL_PATH)
print('Saved summary CSV:  ', SUMMARY_CSV_PATH)
print('Saved per-class CSV:', PER_CLASS_CSV_PATH)
print('Saved run config:   ', RUN_CONFIG_JSON_PATH)


Saved workbook:      /kaggle/working/results/conformal_uncertainty_results/dapm_conformal_reports_all_models.xlsx
Saved summary CSV:   /kaggle/working/results/conformal_uncertainty_results/dapm_conformal_summary_metrics.csv
Saved per-class CSV: /kaggle/working/results/conformal_uncertainty_results/dapm_conformal_per_class_coverage.csv
Saved run config:    /kaggle/working/results/conformal_uncertainty_results/dapm_conformal_run_config.json


# 12.0 — Final Validation

Verifies the workbook was written correctly: checks all required sheet names are present,
confirms the summary has the expected number of rows (3 models × 5 methods = 15), and
validates pixel accounting for map-based methods.

In [23]:
assert EXCEL_PATH.exists(), f'Workbook not found: {EXCEL_PATH}'

wb              = load_workbook(EXCEL_PATH, read_only=True)
existing_sheets = set(wb.sheetnames)

required_prefixes = {
    'Summary_Compact', 'Run_Config',
    'Split_AlexNet',    'Split_GFNet',    'Split_ViT',
    'ClassCond_AlexNet','ClassCond_GFNet','ClassCond_ViT',
    'RC3P_AlexNet',     'RC3P_GFNet',     'RC3P_ViT',
    'Clustered_AlexNet','Clustered_GFNet','Clustered_ViT',
    'RAPS_AlexNet',     'RAPS_GFNet',     'RAPS_ViT',
    'Compare_Clustered','Compare_RAPS',
}

for req in required_prefixes:
    assert any(s.startswith(req) for s in existing_sheets), f'Missing sheet: {req}'

# 3 models × 5 methods
expected_rows = 3 * 5
assert len(summary_compact_df) == expected_rows, (
    f'Expected {expected_rows} rows, got {len(summary_compact_df)}'
)

assert ((summary_compact_df['empirical_coverage'] >= 0) &
        (summary_compact_df['empirical_coverage'] <= 1)).all()
assert (summary_compact_df['avg_set_size'] >= 0).all()

# Full-scene pixel accounting for map-based methods
map_methods = {'SplitConformal', 'ClassConditionalConformal', 'RC3P'}
for out in all_outputs:
    if out['method'] in map_methods:
        px = out['tables']['Pixel Counts']['pixel_count'].sum()
        assert int(px) == H * W, (
            f"Pixel count mismatch for {out['method']} {out['model_name']}: {px} vs {H*W}"
        )

print('Validation passed.')
print('Sheets:', len(existing_sheets))
print('Summary rows:', len(summary_compact_df))
summary_compact_df


Validation passed.
Sheets: 19
Summary rows: 15


,model_name,method,target_coverage,empirical_coverage,avg_set_size,median_set_size,singleton_rate,empty_set_rate,runtime_sec,alpha,lam,n_clusters,mean_per_class_coverage
0,AlexNet,ClassConditionalConformal,0.95,0.883755,10.243321,10.0,0.000000,0.0,1.749501,0.05,NaN,NaN,0.882953
1,GFNet,ClassConditionalConformal,0.95,0.971119,7.190614,6.0,0.000000,0.0,1.739224,0.05,NaN,NaN,0.966703
2,ViT,ClassConditionalConformal,0.95,0.952347,9.392780,10.0,0.000000,0.0,1.703680,0.05,NaN,NaN,0.954420
3,AlexNet,ClusteredConformal,0.95,0.930686,10.681588,11.0,0.000000,0.0,1.998460,0.05,NaN,4.0,0.923000
4,GFNet,ClusteredConformal,0.95,0.959567,7.392058,6.0,0.000000,0.0,2.706980,0.05,NaN,4.0,0.973153
5,ViT,ClusteredConformal,0.95,0.977617,10.933574,11.0,0.000000,0.0,4.623182,0.05,NaN,4.0,0.963253
6,AlexNet,RAPS,0.95,0.963177,12.589170,13.0,0.000000,0.0,0.758604,0.05,0.01,NaN,0.921853
7,GFNet,RAPS,0.95,0.966787,5.402166,5.0,0.000000,0.0,0.501965,0.05,0.01,NaN,0.956993
8,ViT,RAPS,0.95,0.979783,10.257040,10.0,0.000000,0.0,0.475074,0.05,0.01,NaN,0.961313
9,AlexNet,RC3P,0.95,0.917690,10.696029,11.0,0.000000,0.0,1.695208,0.05,NaN,NaN,0.912451
